In [1]:
import os
from typing import TypedDict, List, Dict

from openai import OpenAI


from langsmith.wrappers import wrap_openai
from langsmith import traceable

from langgraph.graph import StateGraph, START, END


In [2]:
from legal_multi_agent.rag.pipeline import legal_rag_retrieve, format_results_for_llm

In [3]:
OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]

MODEL_ID = "qwen/qwen3-235b-a22b-2507"

# OpenRouter با OpenAI SDK سازگار است (base_url)
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# برای ثبت خودکار request/response در LangSmith
openrouter_client = wrap_openai(openrouter_client)

In [4]:
class GraphState(TypedDict, total=False):
    question: str
    rag_results: List[Dict]
    context: str
    answer: str


In [5]:
@traceable(name="retrieve_node")
def retrieve_node(state: GraphState) -> GraphState:
    q = state["question"]

    results = legal_rag_retrieve(
        query=q,
        method="auto",
        top_k=5,
        use_rerank=True,
        verbose=True,
    )
    context = format_results_for_llm(results, include_metadata=True)

    return {
        "rag_results": results,
        "context": context,
    }

In [6]:
@traceable(name="answer_node")
def answer_node(state: GraphState) -> GraphState:
    q = state["question"]
    ctx = state.get("context", "")

    system_msg = (
        "شما یک دستیار حقوقی هستید. فقط بر اساس منابع ارائه‌شده پاسخ بده. "
        "اگر پاسخ از منابع مشخص نیست، صریح بگو اطلاعات کافی در منابع نیست. "
        "ارجاع‌دهی را با ذکر نام قانون و شماره ماده/اصل (اگر در context آمده) انجام بده."
    )

    user_msg = f"""سوال:
{q}

منابع:
{ctx}

وظیفه:
1) پاسخ نهایی را بده.
2) اگر سوال تستی است، گزینه صحیح و دلیل را بگو.
3) هر ادعای حقوقی را به منبع داخل context وصل کن.
"""

    resp = openrouter_client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.2,
    )

    answer = resp.choices[0].message.content
    return {"answer": answer}


In [ ]:
builder = StateGraph(GraphState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("answer", answer_node)

builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "answer")
builder.add_edge("answer", END)

graph = builder.compile()

question = "طبق ماده 190 قانون مدنی، شرایط اساسی صحت معاملات چیست؟"

out = graph.invoke({"question": question})
print(out["answer"])

📝 Query: طبق ماده 190 قانون مدنی، شرایط اساسی صحت معاملات چیست؟...
🎯 روش انتخاب شده: Metadata-aware
🎯 Metadata: law='قانون مدنی' | ماده 190
🔍 فیلتر قانون + شماره (با Range)...
   ✓ 1 سند (law+article_number)
🔍 فیلتر فقط قانون...
   ✓ 10 سند (law_only)
🔍 Semantic fallback...
✅ نهایی: 20 نتیجه
   ✓ Retrieved: 20 documents
🎯 1 نتیجه law+article یافت شد
🔄 Reranking 19 سند دیگر...
   ✓ Reranked: top 5 documents
